# NB_bishop_ch10_convolutional_networks

**Bishop Chapter 10 — Convolutional Neural Networks: CIFAR-10, Filter Visualization, Saliency Maps & Transfer Learning**

This notebook builds a CNN for CIFAR-10, visualizes learned filters, computes saliency maps, demonstrates transfer learning with ResNet50, introduces object detection concepts, and shows a U-Net segmentation architecture.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_10/NB_bishop_ch10_convolutional_networks.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install tensorflow numpy matplotlib

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')

In [ ]:
# Chart styling setup
matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## Part 1: CNN for CIFAR-10

In [ ]:
# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0
y_train = y_train.flatten()
y_test = y_test.flatten()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f'Train: {x_train.shape}, Test: {x_test.shape}')
print(f'Classes: {class_names}')

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = np.random.randint(len(x_train))
    ax.imshow(x_train[idx])
    ax.set_title(class_names[y_train[idx]], fontsize=10)
    ax.axis('off')
fig.suptitle('CIFAR-10 Sample Images', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Build CNN architecture
tf.random.set_seed(42)

cnn_model = tf.keras.Sequential([
    # Block 1
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),
    
    # Block 2
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),
    
    # Block 3
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Dropout(0.25),
    
    # Classifier
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
# Train the CNN
history = cnn_model.fit(
    x_train, y_train,
    epochs=25,
    batch_size=128,
    validation_data=(x_test, y_test),
    verbose=1
)

test_loss, test_acc = cnn_model.evaluate(x_test, y_test, verbose=0)
print(f'\nTest accuracy: {test_acc:.4f}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], color='#1a3a6e', lw=2, label='Train')
axes[0].plot(history.history['val_loss'], color='#cd0000', lw=2, label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('CNN Training Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], color='#1a3a6e', lw=2, label='Train')
axes[1].plot(history.history['val_accuracy'], color='#cd0000', lw=2, label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('CNN Training Accuracy')
axes[1].legend()

fig.tight_layout()
save_fig(fig, 'bishop_ch10_cnn_training')
plt.show()

## Part 2: Visualize Learned Filters

In [ ]:
# Extract first convolutional layer filters
conv1_weights = cnn_model.layers[0].get_weights()[0]  # shape: (3, 3, 3, 32)
print(f'First conv layer filters shape: {conv1_weights.shape}')
print(f'  -> {conv1_weights.shape[3]} filters of size {conv1_weights.shape[0]}x{conv1_weights.shape[1]}x{conv1_weights.shape[2]}')

In [ ]:
# Visualize first 32 filters
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i < conv1_weights.shape[3]:
        filt = conv1_weights[:, :, :, i]
        # Normalize to [0, 1] for display
        filt = (filt - filt.min()) / (filt.max() - filt.min() + 1e-8)
        ax.imshow(filt)
    ax.axis('off')
    ax.set_title(f'F{i}', fontsize=8)
fig.suptitle('Learned Filters: First Convolutional Layer', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch10_filters')
plt.show()

In [ ]:
# Visualize feature maps for a sample image
sample_img = x_test[0:1]  # shape (1, 32, 32, 3)

# Create feature map extractor
layer_outputs = [layer.output for layer in cnn_model.layers if isinstance(layer, tf.keras.layers.Conv2D)]
feature_model = tf.keras.Model(inputs=cnn_model.input, outputs=layer_outputs)
feature_maps = feature_model.predict(sample_img, verbose=0)

print(f'Number of conv layers: {len(feature_maps)}')
for i, fm in enumerate(feature_maps):
    print(f'  Layer {i}: feature map shape {fm.shape}')

In [ ]:
# Plot feature maps from first conv layer
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
fm1 = feature_maps[0][0]  # shape (32, 32, 32)
for i, ax in enumerate(axes.flat):
    if i < fm1.shape[-1]:
        ax.imshow(fm1[:, :, i], cmap='viridis')
    ax.axis('off')
    ax.set_title(f'Ch{i}', fontsize=8)
fig.suptitle('Feature Maps: First Conv Layer', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## Part 3: Saliency Maps

Saliency maps show which pixels most influence the network's prediction by computing the gradient of the output class score with respect to the input image.

In [ ]:
# Compute saliency map
def compute_saliency(model, image, class_idx):
    """Compute vanilla gradient saliency map."""
    image_tensor = tf.cast(tf.constant(image), tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(image_tensor)
        predictions = model(image_tensor)
        target_score = predictions[0, class_idx]
    gradients = tape.gradient(target_score, image_tensor)
    saliency = tf.reduce_max(tf.abs(gradients), axis=-1)[0]  # max across RGB
    return saliency.numpy()

In [ ]:
# Compute saliency for several test images
n_examples = 8
indices = np.random.choice(len(x_test), n_examples, replace=False)

fig, axes = plt.subplots(3, n_examples, figsize=(16, 6))

for col, idx in enumerate(indices):
    img = x_test[idx:idx+1]
    pred = cnn_model.predict(img, verbose=0)
    pred_class = np.argmax(pred)
    true_class = y_test[idx]
    
    saliency = compute_saliency(cnn_model, img, pred_class)
    
    # Original image
    axes[0, col].imshow(x_test[idx])
    axes[0, col].set_title(f'True: {class_names[true_class]}', fontsize=8)
    axes[0, col].axis('off')
    
    # Saliency map
    axes[1, col].imshow(saliency, cmap='hot')
    axes[1, col].set_title(f'Pred: {class_names[pred_class]}', fontsize=8)
    axes[1, col].axis('off')
    
    # Overlay
    axes[2, col].imshow(x_test[idx])
    axes[2, col].imshow(saliency, cmap='hot', alpha=0.5)
    axes[2, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=10)
axes[1, 0].set_ylabel('Saliency', fontsize=10)
axes[2, 0].set_ylabel('Overlay', fontsize=10)

fig.suptitle('Saliency Maps: Where the CNN Looks', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch10_saliency')
plt.show()

## Part 4: Transfer Learning with ResNet50

In [ ]:
# Load ResNet50 pretrained on ImageNet (exclude top classifier)
base_model = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(32, 32, 3)
)
base_model.trainable = False  # Freeze all layers

print(f'ResNet50 layers: {len(base_model.layers)}')
print(f'Trainable params (frozen): {base_model.count_params():,}')

In [ ]:
# Build transfer learning model
transfer_model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(10, activation='softmax')
])

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Use small subset for speed
x_sub = x_train[:5000]
y_sub = y_train[:5000]

transfer_model.summary()

In [ ]:
# Train transfer learning model (feature extraction)
hist_transfer = transfer_model.fit(
    x_sub, y_sub,
    epochs=15,
    batch_size=64,
    validation_data=(x_test, y_test),
    verbose=1
)

print(f'\nTransfer learning test accuracy: {hist_transfer.history["val_accuracy"][-1]:.4f}')

In [ ]:
# Fine-tune: unfreeze last few layers
base_model.trainable = True
# Freeze all but last 10 layers
for layer in base_model.layers[:-10]:
    layer.trainable = False

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # small LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

hist_finetune = transfer_model.fit(
    x_sub, y_sub,
    epochs=10,
    batch_size=64,
    validation_data=(x_test, y_test),
    verbose=1
)

print(f'\nFine-tuned test accuracy: {hist_finetune.history["val_accuracy"][-1]:.4f}')

In [ ]:
# Plot transfer learning results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Combine histories
all_val_acc = hist_transfer.history['val_accuracy'] + hist_finetune.history['val_accuracy']
all_train_acc = hist_transfer.history['accuracy'] + hist_finetune.history['accuracy']
epochs_1 = len(hist_transfer.history['accuracy'])

axes[0].plot(all_train_acc, color='#1a3a6e', lw=2, label='Train')
axes[0].plot(all_val_acc, color='#cd0000', lw=2, label='Val')
axes[0].axvline(epochs_1, color='gray', ls='--', lw=1, label='Fine-tune start')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Transfer Learning: Feature Extraction + Fine-tuning')
axes[0].legend()

# Compare CNN from scratch vs transfer learning
methods = ['CNN (scratch, 25 ep)', 'Transfer (15 ep)', 'Fine-tuned (+10 ep)']
accs = [
    history.history['val_accuracy'][-1],
    hist_transfer.history['val_accuracy'][-1],
    hist_finetune.history['val_accuracy'][-1]
]
bar_colors = ['#1a3a6e', '#DAA520', '#228B22']
axes[1].bar(methods, accs, color=bar_colors, alpha=0.8)
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('CNN from Scratch vs Transfer Learning')
for i, v in enumerate(accs):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=10)

fig.tight_layout()
save_fig(fig, 'bishop_ch10_transfer')
plt.show()

## Part 5: Object Detection Concept (Bounding Boxes)

In [ ]:
# Demonstrate bounding box concept with synthetic data
# Create images with a single MNIST digit placed at a random location
(x_mnist, y_mnist), _ = tf.keras.datasets.mnist.load_data()
x_mnist = x_mnist.astype(np.float32) / 255.0

np.random.seed(42)
canvas_size = 64
n_detection = 200

det_images = []
det_labels = []  # (class, x, y, w, h) normalized

for i in range(n_detection):
    canvas = np.zeros((canvas_size, canvas_size), dtype=np.float32)
    digit_idx = np.random.randint(len(x_mnist))
    digit = x_mnist[digit_idx]
    # Random position
    max_pos = canvas_size - 28
    px = np.random.randint(0, max_pos)
    py = np.random.randint(0, max_pos)
    canvas[py:py+28, px:px+28] = digit
    det_images.append(canvas)
    # Normalized bounding box: (cx, cy, w, h)
    cx = (px + 14) / canvas_size
    cy = (py + 14) / canvas_size
    bw = 28 / canvas_size
    bh = 28 / canvas_size
    det_labels.append([y_mnist[digit_idx], cx, cy, bw, bh])

det_images = np.array(det_images)[..., np.newaxis]  # (N, 64, 64, 1)
det_labels = np.array(det_labels)

print(f'Detection images: {det_images.shape}')
print(f'Labels shape: {det_labels.shape} (class, cx, cy, w, h)')

In [ ]:
# Visualize detection data with bounding boxes
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(det_images[i, :, :, 0], cmap='gray')
    cls, cx, cy, bw, bh = det_labels[i]
    # Convert normalized coords to pixels
    x_tl = (cx - bw/2) * canvas_size
    y_tl = (cy - bh/2) * canvas_size
    w_px = bw * canvas_size
    h_px = bh * canvas_size
    rect = plt.Rectangle((x_tl, y_tl), w_px, h_px,
                         linewidth=2, edgecolor='#cd0000', facecolor='none')
    ax.add_patch(rect)
    ax.set_title(f'Digit: {int(cls)}', fontsize=10)
    ax.axis('off')
fig.suptitle('Object Detection: Digits with Bounding Boxes', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## Part 6: U-Net Architecture for Segmentation (Concept)

In [ ]:
# Build a minimal U-Net architecture
def build_unet(input_shape=(64, 64, 1), n_classes=2):
    inputs = tf.keras.Input(shape=input_shape)
    
    # Encoder
    c1 = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    c1 = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(c1)
    p1 = tf.keras.layers.MaxPooling2D()(c1)
    
    c2 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(p1)
    c2 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(c2)
    p2 = tf.keras.layers.MaxPooling2D()(c2)
    
    # Bottleneck
    c3 = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(p2)
    c3 = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(c3)
    
    # Decoder
    u2 = tf.keras.layers.UpSampling2D()(c3)
    u2 = tf.keras.layers.Concatenate()([u2, c2])  # Skip connection
    c4 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(u2)
    c4 = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(c4)
    
    u1 = tf.keras.layers.UpSampling2D()(c4)
    u1 = tf.keras.layers.Concatenate()([u1, c1])  # Skip connection
    c5 = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(u1)
    c5 = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(c5)
    
    outputs = tf.keras.layers.Conv2D(n_classes, 1, activation='softmax')(c5)
    
    return tf.keras.Model(inputs=inputs, outputs=outputs)

unet = build_unet()
print(f'U-Net parameters: {unet.count_params():,}')
print(f'Input shape: {unet.input_shape}')
print(f'Output shape: {unet.output_shape}')

In [ ]:
# Create synthetic segmentation data (digit vs background)
seg_masks = np.zeros((n_detection, canvas_size, canvas_size, 1), dtype=np.float32)
for i in range(n_detection):
    seg_masks[i, :, :, 0] = (det_images[i, :, :, 0] > 0.1).astype(np.float32)

# Train U-Net briefly
unet.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
unet_hist = unet.fit(det_images[:160], seg_masks[:160].astype(np.int32),
                     epochs=20, batch_size=16,
                     validation_data=(det_images[160:], seg_masks[160:].astype(np.int32)),
                     verbose=0)

print(f'U-Net val accuracy: {unet_hist.history["val_accuracy"][-1]:.4f}')

In [ ]:
# Visualize U-Net predictions
pred_masks = unet.predict(det_images[160:170], verbose=0)
pred_masks = np.argmax(pred_masks, axis=-1)

fig, axes = plt.subplots(3, 5, figsize=(14, 8))
for col in range(5):
    idx = 160 + col
    axes[0, col].imshow(det_images[idx, :, :, 0], cmap='gray')
    axes[0, col].set_title('Input')
    axes[0, col].axis('off')
    
    axes[1, col].imshow(seg_masks[idx, :, :, 0], cmap='gray')
    axes[1, col].set_title('True Mask')
    axes[1, col].axis('off')
    
    axes[2, col].imshow(pred_masks[col], cmap='gray')
    axes[2, col].set_title('Predicted')
    axes[2, col].axis('off')

fig.suptitle('U-Net Segmentation Results', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## CNN Architecture Comparison

In [ ]:
# Compare different CNN architectures
architectures = {
    'Simple (2 conv)': tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=(32,32,3)),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(10, activation='softmax')
    ]),
    'Medium (4 conv)': tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=(32,32,3)),
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(10, activation='softmax')
    ]),
    'MLP (no conv)': tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(32,32,3)),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
}

arch_results = {}
for name, model in architectures.items():
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(x_train[:5000], y_train[:5000], epochs=15, batch_size=64,
                     validation_data=(x_test, y_test), verbose=0)
    arch_results[name] = hist.history
    n_p = model.count_params()
    print(f'{name:>20s}: params={n_p:>8,}, val_acc={hist.history["val_accuracy"][-1]:.4f}')

In [ ]:
# Plot architecture comparison
fig, ax = plt.subplots(figsize=(8, 5))
for (name, hist), c in zip(arch_results.items(), ['#1a3a6e', '#228B22', '#cd0000']):
    ax.plot(hist['val_accuracy'], label=name, color=c, linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Val Accuracy')
ax.set_title('CNN vs MLP on CIFAR-10')
ax.legend()
fig.tight_layout()
plt.show()

## Summary

Key takeaways from Bishop Chapter 10:
- **CNNs** exploit spatial structure through local receptive fields, weight sharing, and pooling.
- **Learned filters** in early layers detect edges and textures; deeper layers capture higher-level features.
- **Saliency maps** reveal which input pixels most influence predictions.
- **Transfer learning** leverages features from pretrained networks (e.g., ImageNet), significantly reducing data requirements.
- **Object detection** extends classification to localization with bounding box regression.
- **U-Net** uses an encoder-decoder architecture with skip connections for pixel-level segmentation.

In [ ]:
print('Notebook complete.')